<a href="https://colab.research.google.com/github/ChenS1313/thelook-ecommerce-analytics-pipeline/blob/ChenS1313-patch-4/thelooker_ecommerce_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##1 Data Loading & Initial Exploration


### 1.1 Loading the relevant tables from **Google BigQuery**

In [ ]:
# Import relevant libraries
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Connect Google account for BigQuery access
auth.authenticate_user()

In [ ]:
# Set up the BigQuery connection and dataset name
client = bigquery.Client(project="project-69dc4deb-819d-43d1-af9")
dataset = "dbt_csinai"

# Create dataframe for each table
df_users = client.list_rows(f"{dataset}.dim_users").to_dataframe()
df_orders = client.list_rows(f"{dataset}.dim_orders").to_dataframe()
df_products = client.list_rows(f"{dataset}.dim_products").to_dataframe()
df_order_items = client.list_rows(f"{dataset}.fct_order_items").to_dataframe()


### 1.2 Checking the data structure to make sure everything loaded correctly

In [ ]:
# Verify table sizes and row counts
print(f"Users table:{df_users.shape}")
print(f"Orders table:{df_orders.shape}")
print(f"Products table:{df_products.shape}")
print(f"Order_items table:{df_order_items.shape}")

In [165]:
# Check column names and data types using .info()
datasets = {
    "Users": df_users,
    "Orders": df_orders,
    "Products": df_products,
    "Order Items": df_order_items
}

for name,df in datasets.items():
   print(f"======== {name} Table Info ========")
   df.info()
   print("\n")

======== Users Table Info ========
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 20 columns):
 #   Column                  Non-Null Count   Dtype        
---  ------                  --------------   -----        
 0   user_id                 100000 non-null  object       
 1   first_name              100000 non-null  object       
 2   last_name               100000 non-null  object       
 3   age                     100000 non-null  Int64        
 4   gender                  100000 non-null  object       
 5   email                   100000 non-null  object       
 6   address                 100000 non-null  object       
 7   city                    100000 non-null  object       
 8   state                   100000 non-null  object       
 9   country                 100000 non-null  object       
 10  postal_code             100000 non-null  object       
 11  latitude                100000 non-null  float64      
 12  longitude 

In [ ]:
# Inspect the data samples
df_users.head()
df_products.head()
df_order_items.head()
df_orders.head()

## 2 Data Formatting

Due to default Pandas display settings when loading data from **Google BigQuery**, we apply light formatting to improve readability without changing the actual data:

- **Dates:** Standardize date formatting.

- **Numeric Columns:** Round prices and costs to 2 decimal places.
This includes casting numeric columns back to `float64` if BigQuery's `NUMERIC` type forced them into Python `object` types during import.

- **Missing Data:** Keep empty cells as Pandas `NaT` / `NaN` / `<NA>` (which match BigQuery `NULL`s).



In [ ]:
# Format all date columns across all tables
dfs_list = [df_users, df_products, df_order_items, df_orders]
for df in dfs_list:
    for col in df.columns:
        # Check if the column is of datetime type
        if df[col].dtype == 'datetime64[us, UTC]':
                df[col] = pd.to_datetime(df[col]).dt.tz_localize(None).astype('datetime64[s]')

In [ ]:
# Convert BigQuery NUMERIC columns from object back to float

# df_users
df_users['lifetime_value'] = pd.to_numeric(df_users['lifetime_value'], errors = 'coerce')
df_users['total_profit'] = pd.to_numeric(df_users['total_profit'], errors = 'coerce')

# df_products
df_products['sale_price'] = pd.to_numeric(df_products['sale_price'], errors = 'coerce')
df_products['cost'] = pd.to_numeric(df_products['cost'], errors = 'coerce')
df_products['total_sales'] = pd.to_numeric(df_products['total_sales'], errors = 'coerce')
df_products['total_profit'] = pd.to_numeric(df_products['total_profit'], errors = 'coerce')


# df_order_items
df_order_items['sale_price'] = pd.to_numeric(df_order_items['sale_price'], errors = 'coerce')
df_order_items['cost'] = pd.to_numeric(df_order_items['cost'], errors = 'coerce')
df_order_items['profit'] = pd.to_numeric(df_order_items['profit'], errors = 'coerce')

# df_orders
df_orders['order_sales'] = pd.to_numeric(df_orders['order_sales'], errors = 'coerce')
df_orders['order_cost'] = pd.to_numeric(df_orders['order_cost'], errors = 'coerce')
df_orders['order_profit'] = pd.to_numeric(df_orders['order_profit'], errors = 'coerce')


In [164]:
# Check that all data types were successfully updated across all datasets using .info()

for name,df in datasets.items():
   print(f"======== {name} Table Info ========")
   df.info()
   print("\n")

======== Users Table Info ========
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 20 columns):
 #   Column                  Non-Null Count   Dtype        
---  ------                  --------------   -----        
 0   user_id                 100000 non-null  object       
 1   first_name              100000 non-null  object       
 2   last_name               100000 non-null  object       
 3   age                     100000 non-null  Int64        
 4   gender                  100000 non-null  object       
 5   email                   100000 non-null  object       
 6   address                 100000 non-null  object       
 7   city                    100000 non-null  object       
 8   state                   100000 non-null  object       
 9   country                 100000 non-null  object       
 10  postal_code             100000 non-null  object       
 11  latitude                100000 non-null  float64      
 12  longitude 

## 3 Deep EDA & Visualizations

### 3.1 Data Overview & Insights

In [166]:
df_users.describe().T

,count,mean,min,25%,50%,75%,max,std
age,100000.0,40.93952,12.0,26.0,41.0,56.0,70.0,17.048016
latitude,100000.0,28.345561,-43.253132,26.198622,35.248808,40.702294,64.865194,22.030017
longitude,100000.0,24.672358,-158.164931,-51.104159,4.51684,116.391848,153.559993,90.115683
total_orders,100000.0,1.25162,0.0,1.0,1.0,2.0,4.0,0.996864
lifetime_value,100000.0,108.286869,0.0,16.0,62.0,151.6,1902.59,136.376211
total_profit,100000.0,56.196505,0.0,8.25,31.23,77.88,1037.56,72.691288
first_order_date,79931,2024-06-11 14:03:14,2019-01-11 04:00:50,2023-04-04 02:19:50,2024-11-17 15:33:46,2025-12-17 03:19:42,2026-07-30 00:41:19,NaN
most_recent_order_date,79931,2025-01-04 19:36:32,2019-01-18 15:18:18,2024-03-17 08:51:04,2025-07-11 13:15:10,2026-03-29 11:20:32,2026-07-30 00:41:19,NaN
user_created_at,100000,2022-11-20 03:45:10,2019-01-02 00:16:00,2020-12-11 12:37:00,2022-11-18 05:32:00,2024-10-28 05:29:45,2026-07-29 19:23:43,NaN


#### 🔍 Insights: `df_users`
* **Age Demographics (`age`):** The average and median age are both **41 years old**, showing a balanced, normal distribution.

* **Customer Retention (`total_orders`):** Over **50% of users have placed only 1 order**. Since this data covers 2019 to 2026, it shows a real **retention challenge** - most customers remain one-time buyers.

* **User Activation (`first_order_date`):** Out of 100,000 registered users, only **79,931 have placed an order**. This leaves ~20,000 "dormant" accounts (non-buyers) that registered but never made a purchase.

* **Lifetime Value (`lifetime_value`):** LTV is **right-skewed**. Most users spend around **\$62.00** (median), but a small group of big spenders pulls the average up to **$108.28**.



In [167]:
df_products.describe().T

,count,mean,std,min,25%,50%,75%,max
sale_price,29120.0,59.220164,65.888927,0.02,24.0,39.99,69.95,999.0
cost,29120.0,28.481889,30.624695,0.01,11.28,19.68,34.44,557.15
total_units_sold,29120.0,6.229293,2.569665,0.0,4.0,6.0,8.0,18.0
total_sales,29120.0,371.864247,479.228573,0.0,120.0,234.0,444.0,11400.0
total_profit,29120.0,192.982502,263.129843,0.0,60.06,116.82,224.5525,6817.2
total_units_returned,29120.0,0.617995,0.784491,0.0,0.0,0.0,1.0,6.0
distribution_center_longitude,29120.0,-88.541508,11.672085,-118.25,-90.0667,-88.0431,-79.9333,-73.7834
distribution_center_latitude,29120.0,34.942039,4.431663,29.7604,30.6944,34.05,39.95,41.8369


#### 🔍 Insights: `df_products`
* **Pricing Strategy (`sale_price` vs. `cost`):** The average sale price ($59.22) is almost **double the average cost** , demonstrating a healthy pricing model.

* **Inventory (`total_units_sold`):**  Some products have **0 units sold**, indicating new catalog additions or inactive "deadstock" products.

* **Product Price Distribution (`sale_price`):**  The price distribution is **right-skewed** (ranging from \$0.02 up to \$999.00). Since 75% of items under $69.95, the store mostly focuses on everyday affordable items, with a few high-end luxury products on top.

In [168]:
df_orders.describe().T

,count,mean,min,25%,50%,75%,max,std
num_of_items,125162.0,1.449298,1.0,1.0,1.0,2.0,4.0,0.805612
created_at,125162,2024-09-22 16:23:47,2019-01-11 04:00:50,2023-09-23 15:58:07,2025-03-18 12:09:39,2026-02-11 06:41:28,2026-07-30 00:41:19,NaN
shipped_at,81303,2024-09-22 19:58:55,2019-01-11 12:14:50,2023-09-23 02:40:31,2025-03-19 10:52:50,2026-02-12 18:24:25,2026-08-01 23:46:43,NaN
delivered_at,43737,2024-09-26 01:41:24,2019-02-02 14:38:48,2023-09-27 16:39:02,2025-03-22 22:42:56,2026-02-15 17:02:04,2026-08-06 10:51:51,NaN
returned_at,12483,2024-09-20 15:04:02,2019-02-04 17:39:48,2023-09-03 11:51:41,2025-03-18 12:06:06,2026-02-11 23:02:34,2026-08-08 18:22:18,NaN
order_sales,125162.0,86.517369,0.02,29.5,55.57,110.0,1892.25,93.497365
order_cost,125162.0,41.618354,0.01,14.28,27.59,53.25,976.49,43.850512
order_profit,125162.0,44.899015,0.01,14.71,28.01,56.76,915.76,50.712226
returned_items_count,125162.0,0.143782,0.0,0.0,0.0,0.0,4.0,0.500337
days_to_ship,81303.0,0.99465,0.0,0.0,1.0,2.0,2.0,0.815089


#### 🔍 Insights: `df_orders`
* **Basket Size (`num_of_items`):** The majority of orders contain only **1 item** (Median: 1, 75th percentile: 2).
* **Returns (`returned_at` & `returned_items_count`):**  While the vast majority of orders have zero returns (`returned_items_count` 75th percentile	= 0), overall **~10% of total orders** (12,483 orders) end up being returned.
* **Order Value (`order_sales`):** The Average Order Value (AOV) stands at 86, but the distribution is right-skewed with high-value outliers reaching up to **$1,892.25**.

* **Shipping (`days_to_ship`):**  Most orders are shipped within **1 day**, with a maximum of **2 days**.
* **Delivery (`days_to_deliver`):** Final delivery to customers takes **3 days on average** (Median: 3.0 days) with a maximum delivery time of **7 days**

In [169]:
df_order_items.describe().T

,count,mean,min,25%,50%,75%,max,std
order_created_at,181397,2024-09-23 07:53:56,2019-01-11 04:00:50,2023-09-24 08:45:34,2025-03-19 02:11:20,2026-02-11 17:45:59,2026-07-30 00:41:19,NaN
item_created_at,181397,2024-09-24 06:52:04,2019-01-11 01:22:24,2023-09-25 05:21:04,2025-03-20 00:09:35,2026-02-12 16:31:05,2026-08-02 23:54:58,NaN
shipped_at,117663,2024-09-24 01:20:29,2019-01-11 12:14:50,2023-09-25 23:03:03,2025-03-21 03:30:52,2026-02-13 16:00:32,2026-08-01 23:46:43,NaN
delivered_at,63360,2024-09-28 12:26:33,2019-02-02 14:38:48,2023-10-01 11:18:08,2025-03-26 16:46:27,2026-02-17 19:48:30,2026-08-06 10:51:51,NaN
returned_at,17996,2024-09-23 12:54:24,2019-02-04 17:39:48,2023-09-11 15:56:49,2025-03-19 03:07:27,2026-02-11 23:40:24,2026-08-08 18:22:18,NaN
sale_price,181397.0,59.696064,0.02,24.5,39.99,69.95,999.0,66.374413
cost,181397.0,28.716221,0.01,11.38,19.84,34.55,557.15,30.812198
profit,181397.0,30.979842,0.01,11.72,20.24,35.9,594.4,36.580634


#### 🔍 Insights: `df_order_items`

* **Item Profitability (`profit`):** Profitability per item is strong at **~52% margin** (calculated as (`profit` / `sale_price`) * 100). Most items make around **\$20.24** (median), but a few high-priced premium products push the overall average up to **\$30.98**, creating a right-skewed distribution.